In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats
%matplotlib
from scipy.integrate import trapezoid
from scipy.optimize import minimize_scalar, brentq
from scipy.stats import poisson


# Honestly just wanted the warnings to disappear from the github, since that kept showing up the entire paths of whoever ran them
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import wimprates as wr

In [ ]:
import sys; sys.path.insert(0, '..')  # Had to be added to now import from folders, before the loader was made with loader.py in the same directory
from src.loader import (
    s2_bin_centers_log, s2_bin_edges,
    k_obs, b_nominal, s_i,
    roi_mask, ROI, s2_energies, dE, bin_starts, recoil_energy_cutoff_kev, response_matrix_nr
)

The Dark Matter model the authors use is a 4 GeV/c² spin-independent elastic WIMP spectrum computed using the `wimprates` package. The signal and all shared variables are pre-computed in `loader.py`. The expected number of DM events per S2 bin is:

$$
s_i = \mathcal{E} \times \sum_j \left. \frac{dR}{dE} \right|_{E_j} \, \Delta E_j \, R_{ji}
$$

In [ ]:
plt.plot(s2_bin_centers_log, s_i, drawstyle='steps-mid')
plt.xlabel("S2 signal [PE]")
plt.ylabel("Expected events per S2 bin")
plt.yscale('log')

Now, to implement the likelihood function itself: (note, for the background itself, taken from the report)

\begin{equation}
\ln \mathcal{L}(\mu_s, \beta)
= \sum_{i=1}^{N}
\left[
k_i \ln \mu_i - \mu_i - \ln(k_i!)
\right]
\end{equation}

Where here we'll take just the 

$$\mu_i = \beta \cdot b_i^\mathrm{nom}$$

where $\mu_i$ is the expected value



In [ ]:
# as noted by the function itself, just the background-only likelihood. 
def log_likelihood_bg_only(beta):
    expected = beta * b_nominal 
    return -np.sum(stats.poisson.logpmf(k_obs, expected))  # negative for minimizer

result = minimize_scalar(log_likelihood_bg_only, bounds=(0.1, 30.0), method='bounded')
beta_hat = result.x

#defining the expected value for just the background:
beta_hat_expected = beta_hat * b_nominal.values

Now, first attempt at graphing it

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

# Observed data as a step histogram
ax.step(s2_bin_edges[:-1], k_obs, where='post', color='black', label='Observed data')

# Background model after fit
ax.step(s2_bin_edges[:-1], beta_hat_expected, where='post', color='blue', 
        linestyle='--', label=f'Background fit (β={beta_hat:.3f})')

ax.set_xscale('log')
ax.set_xlabel('S2 area [PE]')
ax.set_ylabel('Events per bin')
ax.set_xlim(90, 3000)
ax.set_xticks([90, 120, 150, 200, 500, 1000, 3000])
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
ax.legend()
plt.title("Likelihood Function (for backgrounds, without ROI)")
plt.tight_layout()
plt.show()

As seen above, even the background data has higher backgrounds past 1000+, due to further discrenpencies muddying up the results. Now, as the original paper implementing a more proper region of interest (for now the original one in the paper, 165.3, 271.7)

Note, this is also a different cut from the earlier energy cut.

In [ ]:
# roi_mask and ROI are imported from loader
def log_likelihood_bg_only(beta):
    expected = beta * b_nominal.values[roi_mask]
    return -np.sum(stats.poisson.logpmf(k_obs[roi_mask], expected))

result = minimize_scalar(log_likelihood_bg_only, bounds=(0.1, 30.0), method='bounded')
beta_hat = result.x

beta_hat_expected = beta_hat * b_nominal.values

Now, plotting the regular poisson: Honestly, will keep this commented, it doesn't really show anything due to the small bin size

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.step(s2_bin_edges[:-1][roi_mask], k_obs[roi_mask], 
        where='post', color='black', label='Observed data')

ax.step(s2_bin_edges[:-1][roi_mask], beta_hat * b_nominal.values[roi_mask], 
        where='post', color='blue', linestyle='--', 
        label=f'Background fit (β={beta_hat:.3f})')

ax.set_xscale('log')
ax.set_xlabel('S2 area [PE]')
ax.set_ylabel('Events per bin')
ax.set_xticks([165, 200, 271])
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
ax.legend()
plt.title("Background fit within ROI [165.3, 271.7] PE")
plt.tight_layout()
plt.show()

Note, was just trying to visualize the distribution, more properly, use a log-likelihood function.

In [ ]:
beta_values = np.linspace(0.1, 30.0, 500)
log_likes = [-log_likelihood_bg_only(b) for b in beta_values]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(beta_values, log_likes, color='blue')
ax.axvline(beta_hat, color='red', linestyle='--', label=f'β̂ = {beta_hat:.3f}')
ax.set_xlabel('β (background normalization)')
ax.set_ylabel('log L(β)')
ax.set_title('Log-likelihood vs background normalization')
ax.legend()
plt.tight_layout()
plt.show()

Then, following the report, to do the Frequentist uncertainty via the curvature, this formula is used:
$$
    \sigma_{\beta} \approx \sqrt{\left( -\frac{d^2 \ln \mathcal{L}}{d\beta^2} \Big|_{\hat{\beta}} \right)^{-1}}
$$

Note, slight tweak, the equation below uses positive, since it was already set to negative earlier


In [ ]:
h = 1e-4
f = lambda b: log_likelihood_bg_only(b)
d2 = (f(beta_hat + h) - 2*f(beta_hat) + f(beta_hat - h)) / h**2 # This is the second order differential
sigma_beta = np.sqrt(1.0 / d2)
print(f"beta_hat ± sigma_beta = {beta_hat:.4f} ± {sigma_beta:.4f}")

In [ ]:
print('test for part 3-JamesX')


#We first define the Prior p(beta)
beta_min, beta_max = 0.1, 30.0 #0.1 and 30 because same scale already used
prior_val = 1.0 / (beta_max-beta_min)
print(prior_val)





#Calculate the unnormalized posterior
beta_grid=np.linspace(beta_min,beta_max,1000)
unnormalized_posterior = []
for b in beta_grid:
    log_like=-log_likelihood_bg_only(b) 
    unnormalized_posterior.append(np.exp(log_like)*prior_val)
unnormalized_posterior=np.array(unnormalized_posterior)
evidence=trapezoid(unnormalized_posterior,x=beta_grid)
#We now get the normaised posterior distribution
posterior=unnormalized_posterior/evidence



#We check the uncertainties ----> mean and credible interval which should be (68%)
mean_beta = trapezoid(beta_grid * posterior, x=beta_grid)

print(mean_beta)


In [ ]:
cumulative_posterior = np.cumsum(posterior) * (beta_grid[1] - beta_grid[0])
low_bound = beta_grid[np.searchsorted(cumulative_posterior, 0.16)]
high_bound = beta_grid[np.searchsorted(cumulative_posterior, 0.84)]

print(f"Bayesian Mean β: {mean_beta:.4f}")
print(f"68% Credible Interval: [{low_bound:.4f}, {high_bound:.4f}]")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(beta_grid, posterior, label='Posterior $P(\\beta | \\text{data})$', color='darkgreen')
plt.axvline(mean_beta, color='orange', linestyle='--', label=f'Mean $\\beta$ = {mean_beta:.3f}')
plt.fill_between(beta_grid, posterior, where=(beta_grid >= low_bound) & (beta_grid <= high_bound), 
                 alpha=0.3, color='green', label='68% Credible Interval')
plt.xlabel('$\\beta$ (Background Normalization)')
plt.ylabel('Probability Density')
plt.title('Bayesian Posterior Distribution for Background Fit')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Now, do actually implement the check with the signal test. Luckily, most of the work was done earlier, so just implementing the variable $\mu_s$ and defining the a new log likelihood changes this.

In [ ]:
# Extend background-only likelihood to include signal component
# Reuses: roi_mask, b_nominal, s_i, k_obs, beta_values all defined above

mu_s = 10  # signal strength parameter, tweak to see different results!

def log_likelihood_full(beta, mu_s=mu_s):
    expected = beta * b_nominal.values[roi_mask] + mu_s * s_i[roi_mask]
    expected = np.clip(expected, 1e-12, None)
    return -np.sum(stats.poisson.logpmf(k_obs[roi_mask], expected))

result_full = minimize_scalar(lambda b: log_likelihood_full(b, mu_s),
                              bounds=(0.1, 30.0), method='bounded')
beta_hat_full = result_full.x

# Overlay signal+background log-likelihood on the existing beta scan
log_likes_full = [-log_likelihood_full(b, mu_s) for b in beta_values]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(beta_values, log_likes,       color='blue',   label='Background only')
ax.plot(beta_values, log_likes_full,  color='orange', label=f'Signal + background (μ_s={mu_s})')
ax.axvline(beta_hat,      color='red',        linestyle='--', label=f'β̂ (bg only) = {beta_hat:.3f}')
ax.axvline(beta_hat_full, color='darkorange', linestyle='--', label=f'β̂ (signal+bg) = {beta_hat_full:.3f}')
ax.set_xlabel('β (background normalization)')
ax.set_ylabel('log L(β)')
ax.set_title(f'Log-likelihood vs β within ROI (μ_s = {mu_s})')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Systematic uncertainties — defined once, used by both the mass scan and the single-mass limit below
sigma_agg_signal = np.sqrt(
    0.05**2  +   # electron lifetime
    0.025**2 +   # S2 gain g2
    0.12**2  +   # S2 width cut efficiency (dominant for 4 GeV NR)
    0.08**2  +   # radial cut efficiency
    0.07**2  +   # pileup rejection cuts
    0.03**2      # gas event cut
)                # gives sigma_agg_s ~ 16.5%

sigma_agg_background = np.sqrt(
    0.05**2  +   # electron lifetime
    0.025**2     # S2 gain g2
)                # gives sigma_agg_b ~ 5.6%

In [ ]:
# DM masses to scan 
dm_masses = np.array([3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0])  # GeV/c^2

sigma_ref    = 1e-45   # cm^2 — reference cross section
sigma_uppers = []      # will store upper limit at each mass

# Loop over masses and compute upper limit at each
for mw in dm_masses:

    # signal template for this mass
    rate_kev = wr.rate_wimp_std(
        es=s2_energies,
        mw=mw,
        sigma_nucleon=sigma_ref
    )
    rate = rate_kev * dE
    rate = rate * 0.97678  # efficiency factor from paper

    # apply energy cutoff
    rate_cut = rate.copy()
    cutoff_idx = (bin_starts < recoil_energy_cutoff_kev).sum() - 1
    rate_cut[:cutoff_idx] = 0
    suppress = (recoil_energy_cutoff_kev - bin_starts[cutoff_idx]) / bin_starts[cutoff_idx]
    suppress = np.clip(suppress, 0, 1)
    rate_cut[cutoff_idx] *= 1 - suppress

    # project through response matrix
    s_i_mass = rate_cut @ response_matrix_nr

    # roi_mask imported from loader — fixed ROI, ideally optimised per mass
    n_obs_mass    = int(k_obs[roi_mask].sum())
    mu_b_nom_mass = b_nominal.values[roi_mask].sum()
    mu_s_nom_mass = s_i_mass[roi_mask].sum()

    # conservative expectations
    mu_s_cons = max(mu_s_nom_mass * (1 - 1.28 * sigma_agg_signal),     0.0)
    mu_b_cons = max(mu_b_nom_mass * (1 - 1.28 * sigma_agg_background), 0.0)

    # Poisson upper limit 
    def poisson_cdf_minus_010(mu_s_val):
        mu_total = mu_s_val * mu_s_cons + mu_b_cons
        return poisson.cdf(n_obs_mass, mu_total) - 0.10

    try:
        mu_s_up  = brentq(poisson_cdf_minus_010, 0, 1e10)
        # apply floor (limit)
        if mu_s_up * mu_s_cons < 2.3:
            mu_s_up = 2.3 / mu_s_cons if mu_s_cons > 0 else np.nan
        sigma_up = mu_s_up * sigma_ref
    except ValueError:
        sigma_up = np.nan

    sigma_uppers.append(sigma_up)
    print(f"mw = {mw:.1f} GeV/c^2 | mu_s_nom = {mu_s_nom_mass:.5f} | "
          f"n_obs = {n_obs_mass} | sigma_up = {sigma_up:.3e} cm^2")

sigma_uppers = np.array(sigma_uppers)

# Plotting
fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(dm_masses, sigma_uppers, 
        color='black', linewidth=2, 
        label='This work (90% CL)')
ax.fill_between(dm_masses, sigma_uppers, 
                sigma_uppers.max() * 10,
                alpha=0.2, color='gray')

ax.set_yscale('log')
ax.set_xscale('linear')
ax.set_xlabel(r'Dark matter mass $m_\chi$ [GeV/$c^2$]', fontsize=12)
ax.set_ylabel(r'$\sigma$ [cm$^2$] (SI, spin-independent)', fontsize=12)
ax.set_title('90% CL upper limit on SI DM-nucleon cross section\n'
             '(4 GeV/$c^2$ spin-independent NR, S2-only)', fontsize=11)
ax.set_xlim(2.5, 6.5)
ax.set_ylim(1e-44, 1e-40)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#observed events and background in ROI
#n_obs = the observed count in the ROI
n_obs    = int(k_obs[roi_mask].sum())        
mu_b_nom = b_nominal.values[roi_mask].sum()  
mu_s_nom = s_i[roi_mask].sum()              

# Conservative 10th percentile expectations (sigma_agg_* defined above)
#  s_cons    = s    * (1 - 1.28 * sigma_agg_s)
#  mu_b_cons = mu_b * (1 - 1.28 * sigma_agg_b)
#
# 1.28 is the 90th percentile of the standard normal distribution
mu_s_conservative = mu_s_nom * (1 - 1.28 * sigma_agg_signal)
mu_b_conservative = mu_b_nom * (1 - 1.28 * sigma_agg_background)
mu_s_conservative = max(mu_s_conservative, 0.0)  
mu_b_conservative = max(mu_b_conservative, 0.0) 

# Poisson CDF condition
# This is the left-hand side of Eq.(eq:poisson_ul) minus 0.10:
#
#   f(mu_s) = sum_{k=0}^{n_obs} 
#             e^{-(mu_s*s_cons + mu_b_cons)} * (mu_s*s_cons + mu_b_cons)^k / k!
#             - 0.10
#
# We want to find mu_s_up such that f(mu_s_up) = 0, here the Poisson CDF equals exactly 0.10.
# equivalent to finding where values of mu_s > mu_s_up are excluded at 90% CL.

def poisson_cdf_minus_010(mu_s_val):
    # mu_total is the total expected count at this mu_s value
    mu_total = mu_s_val * mu_s_conservative + mu_b_conservative
    return poisson.cdf(n_obs, mu_total) - 0.10

# Solve numerically using Brent's method
# brentq finds the root of poisson_cdf_minus_010 in [0, 1e9]:
# the value mu_s_up where the CDF = 0.10 exactly — the 90% CL upper limit on mu_s.

try:
    mu_s_upper  = brentq(poisson_cdf_minus_010, 0, 1e9)
    sigma_upper = mu_s_upper * 1e-45  # Eq.(sigma_up = mu_s_up * sigma_ref)

    print(f"\n=== 90% CL Upper Limit ===")
    print(f"mu_s_up                    = {mu_s_upper:.4f}")
    print(f"Expected signal at limit   = {mu_s_upper * mu_s_conservative:.4f} events")
    print(f"sigma_up = mu_s_up * sigma_ref = {sigma_upper:.3e} cm^2")

except ValueError as e:
    print(f"brentq failed: {e}")

# Apply the paper's floor: "never exclude signals with < 2.3 expected events"
# 2.3 is the 90% CL Poisson upper limit for n_obs=0 and mu_b=0. Prevents the limit
# from being artificially strong in bins with very low background.

mu_s_upper_counts = mu_s_upper * mu_s_conservative  # expected signal events at limit

if mu_s_upper_counts < 2.3:
    mu_s_upper  = 2.3 / mu_s_conservative
    sigma_upper = mu_s_upper * 1e-45
    print(f"\nFloor applied: expected signal was {mu_s_upper_counts:.4f} < 2.3 events")

# The 90% CL upper limit on the SI DM-nucleon cross section for a 4 GeV/c^2 WIMP, ROI = 165.3-271.7 PE.
print(f"\n=== Final Result ===")
print(f"mu_s         < {mu_s_upper:.4f}  (dimensionless signal strength)")
print(f"sigma        < {sigma_upper:.3e} cm^2  (90% CL)")
print(f"Signal events at limit: {mu_s_upper * mu_s_conservative:.4f}")